# Bike Rental Demand — Random Forest

A nonlinear tree-based model for bicycle-rental demand, including model selection, time-aware validation, error analysis and feature-importance diagnostics.

> **Portfolio note:** This notebook is a curated version of completed MSc coursework. The analytical code and saved outputs come from the original work; presentation, headings, file paths and explanatory text have been cleaned for portfolio use. The dataset itself is not redistributed here.


**IMPORTS**


In [3]:
# Importing the libraries
import numpy as np # for array operations
import pandas as pd # for working with DataFrames
import requests, io # for HTTP requests and I/O commands
import matplotlib.pyplot as plt # for data visualization
import holoviews as hv
import os
from IPython.display import display, HTML
%matplotlib inline

# scikit-learn modules
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor  # or RandomForestRegressor for regression
from sklearn.tree import DecisionTreeRegressor  # Use DecisionTreeRegressor for regression tasks
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, MinMaxScaler, RobustScaler
from sklearn.model_selection import cross_val_score, BaseCrossValidator, GridSearchCV, train_test_split
from sklearn import metrics
from sklearn.metrics import r2_score, root_mean_squared_error, mean_squared_error
from sklearn.inspection import PartialDependenceDisplay
from sklearn.impute import SimpleImputer
from sklearn.base import clone


**CONSTANTS**


In [29]:
FILE_PATH = 'data/bike_output.csv'


**OPTIONS**


**METHODS**


In [35]:
import numpy as np
import pandas as pd
from sklearn.model_selection import BaseCrossValidator
from sklearn.utils.validation import _num_samples

class ExpandingWindowSplit(BaseCrossValidator):
    """
    Expanding Window Cross-Validator for Time Series Data based on Months.

    Splits the data into training and test sets based on sorted month periods.
    The training set size expands in each split, incorporating previous test periods.

    Parameters
    ----------
    month_series : pd.Series
        A pandas Series where the index corresponds to the sample index in X/y,
        and the values are comparable month identifiers (e.g., pd.Period, strings 'YYYY-MM').
        Must be sorted chronologically by month value if creating internally.

    initial_train_months : int
        The number of months to include in the first training split.

    test_months : int, default=1
        The number of months to include in each test split.

    gap_months : int, default=0
        The number of months to skip between the end of the training set
        and the beginning of the test set.

    max_splits : int or None, default=None
        The maximum number of splits to generate. If None, all possible splits
        according to the parameters will be generated.

    Attributes
    ----------
    unique_months_ : np.ndarray
        Sorted unique month values found in `month_series`.

    n_splits_ : int
        The number of splits generated.
    """
    def __init__(self, month_series, initial_train_months, test_months=1, gap_months=0, max_splits=None):
        if not isinstance(month_series, pd.Series):
            raise ValueError("month_series must be a pandas Series.")
        if initial_train_months < 1:
            raise ValueError("initial_train_months must be at least 1.")
        if test_months < 1:
            raise ValueError("test_months must be at least 1.")
        if gap_months < 0:
            raise ValueError("gap_months cannot be negative.")

        self.month_series = month_series
        self.initial_train_months = initial_train_months
        self.test_months = test_months
        self.gap_months = gap_months
        self.max_splits = max_splits

        # Store unique sorted months - crucial step
        self.unique_months_ = np.sort(month_series.unique())
        self.n_total_months_ = len(self.unique_months_)

        if self.initial_train_months + self.gap_months + self.test_months > self.n_total_months_:
            raise ValueError(
                f"Not enough months ({self.n_total_months_}) for the first split "
                f"with initial_train_months={initial_train_months}, "
                f"gap_months={gap_months}, and test_months={test_months}."
            )

        # Pre-calculate the number of splits
        self.n_splits_ = self._calculate_n_splits()

    def _calculate_n_splits(self):
        """Calculates the total number of splits possible."""
        max_possible_splits = 0
        current_train_end_idx = self.initial_train_months
        while True:
            test_start_idx = current_train_end_idx + self.gap_months
            test_end_idx = test_start_idx + self.test_months
            if test_end_idx <= self.n_total_months_:
                max_possible_splits += 1
                # Move to the next potential split start
                # Training data expands by the size of the test set
                current_train_end_idx += self.test_months
            else:
                break # Cannot form another split

        if self.max_splits is not None:
            return min(max_possible_splits, self.max_splits)
        else:
            return max_possible_splits

    def get_n_splits(self, X=None, y=None, groups=None):
        """Returns the number of splitting iterations in the cross-validator."""
        return self.n_splits_

    def split(self, X, y=None, groups=None):
        """Generate indices to split data into training and test set."""
        n_samples = _num_samples(X)
        indices = np.arange(n_samples) # Use simple range if index isn't meaningful
        # Or, if X is a DataFrame with a meaningful index that month_series shares:
        # indices = self.month_series.index.to_numpy() # Ensure alignment

        current_train_end_idx = self.initial_train_months
        splits_generated = 0

        while splits_generated < self.n_splits_:
            test_start_idx = current_train_end_idx + self.gap_months
            test_end_idx = test_start_idx + self.test_months

            # Check boundary condition again (safety)
            if test_end_idx > self.n_total_months_:
                break

            train_months = self.unique_months_[0:current_train_end_idx]
            test_months_fold = self.unique_months_[test_start_idx:test_end_idx]

            # Get boolean masks for selecting indices
            train_mask = self.month_series.isin(train_months)
            test_mask = self.month_series.isin(test_months_fold)

            # Select indices based on the mask
            train_idx = indices[train_mask]
            test_idx = indices[test_mask]

            # Ensure test indices are chronologically after train indices
            # (Should be guaranteed by month selection, but good check)
            # if train_idx.size > 0 and test_idx.size > 0 and train_idx.max() >= test_idx.min():
            #    warnings.warn("Found overlapping or non-sequential train/test indices.", UserWarning)

            yield train_idx, test_idx

            # Expand the training window for the next iteration
            current_train_end_idx += self.test_months
            splits_generated += 1


**LOAD DATA**


In [38]:
df_bike_model = pd.read_csv(FILE_PATH)
df_bike_model['dteday'] = pd.to_datetime(df_bike_model['dteday'])
df_bike_model.sort_values('dteday', inplace=True)


FileNotFoundError: [Errno 2] No such file or directory: './Data/Clean/bike_output.csv'

In [25]:
df_bike_model.tail()


NameError: name 'df_bike_model' is not defined

**PREPARING DATA FOR MODELING**


In [ ]:
# Create the 'year_month' column as before
df_bike_model['year_month'] = df_bike_model['dteday'].dt.to_period('M')
month_series = df_bike_model['year_month'] # Ensure index matches X
# Sort the dataframe by 'year_month' to ensure chronological order
df_bike_model.sort_values('year_month', inplace=True)

df_bike_model['time_idx'] = df_bike_model.index
# 3. Cyclical Features
# Using existing 'mnth' column
df_bike_model['mnth_sin'] = np.sin(2 * np.pi * df_bike_model['mnth'] / 12)
df_bike_model['mnth_cos'] = np.cos(2 * np.pi * df_bike_model['mnth'] / 12)

# Using existing 'weekday' column (Ensure it's 0-6 or 1-7 consistently)
# Assuming 0=Monday, 6=Sunday as per dt.weekday standard
# If your 'weekday' column follows a different convention, adjust the division factor (e.g., use 7 if it's 1-7)
df_bike_model['weekday_sin'] = np.sin(2 * np.pi * df_bike_model['weekday'] / 7)
df_bike_model['weekday_cos'] = np.cos(2 * np.pi * df_bike_model['weekday'] / 7)

# 4. Lag Features (Using shift(1) as primary, keep shift(2) based on your comment, correct shift(7))
# shift(2): Day before yesterday's count (as per your comment)
df_bike_model['cnt_lag2'] = df_bike_model['cnt'].shift(2)
# shift(7): Count from the same day last week
df_bike_model['cnt_lag7'] = df_bike_model['cnt'].shift(7)

# 5. Rolling Window Features (Based on shift(1) to use most recent complete day)
# Use closed='left' to ensure the window includes data up to, but not including, the current day
df_bike_model['cnt_roll3_lag2'] = df_bike_model['cnt'].shift(2).rolling(window=3, closed='left').mean()
df_bike_model['cnt_roll7_lag2'] = df_bike_model['cnt'].shift(2).rolling(window=7, closed='left').mean()
df_bike_model['cnt_roll14_lag2'] = df_bike_model['cnt'].shift(2).rolling(window=14, closed='left').mean()


In [ ]:
# unique_months is a sorted array of the unique 'year_month' values in the dataset.
unique_months = np.array(sorted(df_bike_model['year_month'].unique()))
print("Unique months:", unique_months)
print("Total months:", len(unique_months))


In [ ]:
df_bike_model.info()


In [ ]:
df_bike_model.head()


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
# from sklearn.impute import SimpleImputer # Uncomment if needed
# from sklearn.preprocessing import StandardScaler # Uncomment if needed
from sklearn.model_selection import GridSearchCV

# Assume X and y are prepared (features and target)
# Ensure X and y use the same index as month_series
feature_cols = [
    'season', 'yr', 'mnth', 'holiday', 'weekday', 'workingday', 'weathersit', 'atemp', 'hum', 'windspeed', # RAW FEATURES
    'mnth_sin', 'mnth_cos', # months as sin / cos
    'cnt_roll3_lag2', 'cnt_roll7_lag2', 'cnt_roll14_lag2',
]
target_col = 'cnt'

X = df_bike_model[feature_cols]
y = df_bike_model[target_col]


In [ ]:
X.head()


In [ ]:
y.head()


In [ ]:
# Create a pipeline for the Random Forest
pipeline_rf = Pipeline([
    # Add preprocessing steps here if needed (e.g., Scaler, Imputer)
    # ('scaler', StandardScaler()),
    ('imputer', SimpleImputer(strategy='median')), # Use median or mean
    ('rf', RandomForestRegressor(random_state=42))
])

# Define a parameter grid for the Random Forest
param_grid_rf = {
    # 'imputer__strategy': ['mean', 'median'], # Usually stick to one for simplicity
    'rf__n_estimators': [100, 200], # More estimators can help, but increase time
    'rf__max_depth': [10, 15, None], # Control complexity
    'rf__min_samples_split': [5, 10], # Regularization
    'rf__min_samples_leaf': [3, 5], # Regularization
    'rf__max_features': ['sqrt', 0.7] # Consider subset of features per split ('sqrt', 'log2', or float fraction)
}

# Instantiate the Custom Cross-Validator (Using RemainderTest version)
# Ensure the ExpandingWindowSplitRemainderTest class definition is available
cv = ExpandingWindowSplit(
    month_series=month_series,
    initial_train_months=14,   # Start with 14 months of training data
    gap_months=0,             # No gap between train and test
    max_splits=5,             # Limit to 5 folds (or set to None for all possible)
)

# --- Run GridSearchCV ---

# Set up GridSearchCV
gs = GridSearchCV(
    estimator=pipeline_rf,
    param_grid=param_grid_rf,
    cv=cv,  # Use the custom time series CV
    scoring={'r2': 'r2', 'rmse': 'neg_root_mean_squared_error'},
    refit='r2', # Or 'rmse' if preferred
    n_jobs=-1,  # Use all available CPU cores
    verbose=1   # Show GridSearchCV progress
)

print("\nStarting GridSearchCV Fit...")
# Fit (Make sure X and y indices align with month_series if they are DataFrames/Series)
# If X/y were converted to numpy, ensure month_series aligns row-wise
gs.fit(X, y) # Can use pandas DataFrame directly if indices align

print("\nGridSearchCV Fit Complete.")

# --- Display Results ---
print("\nBest Parameters Found:")
print(gs.best_params_)
print(f"\nBest CV R² Score ({gs.refit}): {gs.best_score_:.4f}")

# Optionally, examine detailed results
print("\nFull CV Results:")
cv_results_df = pd.DataFrame(gs.cv_results_)
print(cv_results_df[['params', 'mean_test_r2', 'mean_test_rmse', 'rank_test_r2']])


## **EVALUATION**


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.base import clone

fold_results = []

# gs is your GridSearchCV object that you've already fit
# cv is your custom FiveMonthSplits instance
for fold_num, (train_idx, test_idx) in enumerate(cv.split(X, y), start=1):
    # Split the data into training and testing subsets for this fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    # Clone the best estimator from GridSearchCV (same hyperparameters) 
    # so we can train it on just this fold’s training data
    model = clone(gs.best_estimator_)
    model.fit(X_train, y_train)
    
    # Predict on the training subset
    y_pred_train = model.predict(X_train)
    # Predict on the test subset
    y_pred_test = model.predict(X_test)
    
    # Compute metrics on the training set
    r2_train = r2_score(y_train, y_pred_train)
    rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
    mae_train = mean_absolute_error(y_train, y_pred_train)
    
    # Compute metrics on the test set
    r2_test = r2_score(y_test, y_pred_test)
    rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
    mae_test = mean_absolute_error(y_test, y_pred_test)
    
    # Store both training and test metrics for this fold
    fold_results.append({
        'fold': fold_num,
        'r2_train': r2_train,
        'rmse_train': rmse_train,
        'mae_train': mae_train,
        'r2_test': r2_test,
        'rmse_test': rmse_test,
        'mae_test': mae_test
    })

# Convert results to a DataFrame for an overview
results_df = pd.DataFrame(fold_results)
print(results_df)


Overfitting in earlier folds with small training sets is quite normal in a time-series CV. It doesn’t necessarily signal a problem if your final production model is trained on the larger dataset. However, it does remind you that, early on (when less data is available), the model might be less reliable unless you address overfitting through feature engineering or regularization.


**ACTUAL VS PREDICTED**


In [ ]:
import matplotlib.pyplot as plt

# Actual vs. Predicted on the TRAIN set
plt.figure()
plt.plot(range(len(y_train)), y_train, marker='o', label='Train Actual')
plt.plot(range(len(y_train)), y_pred_train, marker='x', label='Train Predicted')
plt.title("TRAIN SET: Actual vs. Predicted")
plt.xlabel("Index (Train Samples)")
plt.ylabel("Target Value")
plt.legend()
plt.show()


**TRAIN SET RESIDUALS**


In [ ]:
# Residuals for the TRAIN set
residuals_train = y_train - y_pred_train

plt.figure()
plt.scatter(range(len(y_train)), residuals_train, marker='o')
plt.axhline(y=0, color='r', linestyle='--')
plt.title("TRAIN SET: Residuals")
plt.xlabel("Index (Train Samples)")
plt.ylabel("Residual (Actual - Predicted)")
plt.show()


**TEST SET: ACTUAL VS. PREDICTED**


In [ ]:
# Make sure you have y_test defined (the actual target values for your test set)
plt.figure()
plt.plot(range(len(y_test)), y_test, marker='o', label='Test Actual')
plt.plot(range(len(y_test)), y_pred_test, marker='x', label='Test Predicted')
plt.title("TEST SET: Actual vs. Predicted")
plt.xlabel("Index (Test Samples)")
plt.ylabel("Target Value")
plt.legend()
plt.show()


**TEST SET RESIDUALS**


In [ ]:
# Residuals for the TEST set
residuals_test = y_test - y_pred_test

plt.figure()
plt.scatter(range(len(y_test)), residuals_test, marker='o')
plt.axhline(y=0, color='r', linestyle='--')
plt.title("TEST SET: Residuals")
plt.xlabel("Index (Test Samples)")
plt.ylabel("Residual (Actual - Predicted)")
plt.show()
